In [1]:
from typing import Any, Dict, List, Optional, Tuple

import hydra
import lightning as L
import rootutils
import torch
from lightning import Callback, LightningDataModule, LightningModule, Trainer
from lightning.pytorch.loggers import Logger
from omegaconf import DictConfig

In [2]:
cd ..

/work/hpc/spine-segmentation


In [3]:
!export CUDA_VISIBLE_DEVICES=3

In [4]:
path = "/work/hpc/spine-segmentation/logs/train/runs/attention-unet-v2/checkpoints/epoch_271_v2.ckpt"
new_path = "/work/hpc/spine-segmentation/logs/train/runs/2024-10-16_19-48-52/checkpoints/epoch_223.ckpt"

In [5]:
import rootutils
rootutils.setup_root(search_from="/work/hpc/spine-segmentation/", indicator="setup.py", pythonpath=True)

PosixPath('/work/hpc/spine-segmentation')

checkpoint = torch.load(path)

In [6]:
pwd

'/work/hpc/spine-segmentation'

In [7]:
with hydra.initialize(version_base="1.3", config_path="../configs", ):
    cfg = hydra.compose(config_name='train.yaml')
    print(cfg)

{'task_name': 'train', 'tags': ['dev'], 'train': True, 'test': True, 'ckpt_path': None, 'seed': None, 'data': {'transform_train': {'_target_': 'monai.transforms.Compose', 'transforms': [{'_target_': 'monai.transforms.LoadImaged', 'keys': ['image', 'label']}, {'_target_': 'src.data.transforms.array.ConvertToMultiChannelBasedOnSpiderClassesdSemantic', 'keys': 'label', 'div': 100, 'labels': [0, 1, 2, 3]}, {'_target_': 'monai.transforms.EnsureChannelFirstd', 'keys': 'image'}, {'_target_': 'monai.transforms.Spacingd', 'keys': ['image', 'label'], 'pixdim': '${data.spacing}', 'mode': [3, 'nearest']}, {'_target_': 'monai.transforms.RandAffined', 'keys': ['image', 'label'], 'prob': 0.15, 'scale_range': [0.1, 0.15, 0.15], 'rotate_range': [0.1, 0.3, 0.3], 'padding_mode': 'zeros', 'mode': [3, 'nearest']}, {'_target_': 'monai.transforms.NormalizeIntensityd', 'keys': 'image', 'nonzero': True, 'channel_wise': True}, {'_target_': 'monai.transforms.OneOf', 'transforms': [{'_target_': 'monai.transforms.

In [11]:
module = hydra.utils.instantiate(cfg.model)

[[1. 0. 0.]
 [0. 9. 0.]
 [0. 0. 9.]]
141.7464895115018
Roi: [ 32 256 256]
Ratio [0 2 2]
Iso stride [[1 2 2]
 [1 2 2]]
Roi [32. 64. 64.]
Bottleneck 2
Zoom 4
Total Stride [[2 2 2]
 [2 2 2]
 [2 2 2]
 [2 2 2]
 [1 2 2]
 [1 2 2]]
[ 16  32  64 128 256 362 512]
{'spatial_dims': 3, 'in_channels': 1, 'out_channels': 4, 'kernel_size': 3, 'up_kernel_size': 5, 'channels': [16, 32, 64, 128, 256, 362, 512], 'strides': [[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [1, 2, 2], [1, 2, 2]], 'dropout': 0.5, '_target_': 'monai.networks.nets.AttentionUnet'}


/work/hpc/miniconda3/envs/mri/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.


In [12]:
module.__class__

src.models.spider_semantic_module_test.SpiderLitModule

In [13]:
net = hydra.utils.instantiate(cfg.model.net)

Roi: [ 32 256 256]
Ratio [0 2 2]
Iso stride [[1 2 2]
 [1 2 2]]
Roi [32. 64. 64.]
Bottleneck 2
Zoom 4
Total Stride [[2 2 2]
 [2 2 2]
 [2 2 2]
 [2 2 2]
 [1 2 2]
 [1 2 2]]
[ 16  32  64 128 256 362 512]
{'spatial_dims': 3, 'in_channels': 1, 'out_channels': 4, 'kernel_size': 3, 'up_kernel_size': 5, 'channels': [16, 32, 64, 128, 256, 362, 512], 'strides': [[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [1, 2, 2], [1, 2, 2]], 'dropout': 0.5, '_target_': 'monai.networks.nets.AttentionUnet'}


In [14]:
module = module.__class__.load_from_checkpoint("/work/hpc/spine-segmentation/logs/train/runs/2024-10-16_19-48-52/checkpoints/epoch_223.ckpt", net=net)

In [16]:
ema = module.ema

In [17]:
print(ema)

In [18]:
ema.num_updates

0

In [19]:
ema.shadow_params

[tensor([[[[[ 0.1327,  0.1473,  0.1596],
            [ 0.1072, -0.0238,  0.0461],
            [-0.0866, -0.1819,  0.1452]],
 
           [[-0.1852,  0.0522, -0.0874],
            [ 0.0370, -0.1631, -0.0217],
            [-0.0700,  0.1472,  0.1864]],
 
           [[ 0.0574,  0.1584, -0.0870],
            [ 0.1077,  0.1424, -0.0707],
            [-0.0948, -0.0357,  0.0566]]]],
 
 
 
         [[[[ 0.1691,  0.0801, -0.1187],
            [-0.1107,  0.1274, -0.0032],
            [-0.0979,  0.0562, -0.1746]],
 
           [[ 0.1510,  0.1128, -0.0625],
            [ 0.0172, -0.1906, -0.0647],
            [ 0.0117, -0.0114,  0.1904]],
 
           [[ 0.1473, -0.1452,  0.0279],
            [ 0.1011,  0.1441,  0.1780],
            [ 0.0775, -0.1831,  0.0855]]]],
 
 
 
         [[[[-0.0428,  0.0594,  0.1861],
            [ 0.0974, -0.0795, -0.0025],
            [ 0.0508,  0.0488,  0.1612]],
 
           [[-0.1340,  0.1540,  0.1871],
            [ 0.1677, -0.0104, -0.1337],
            [ 0.1589, -0

In [8]:
datamodule = hydra.utils.instantiate(cfg.data)
datamodule.setup()

/work/hpc/miniconda3/envs/mri/lib/python3.10/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.18 (you have 1.4.14). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [9]:
val_dat = iter(datamodule.val_dataloader())

In [10]:
batch = next(val_dat)

In [11]:
label = batch['label'].detach().cpu().numpy()

In [16]:
batch['label_meta_dict']

{'sizeof_hdr': tensor([348], dtype=torch.int32),
 'extents': tensor([0], dtype=torch.int32),
 'session_error': tensor([0], dtype=torch.int16),
 'dim_info': tensor([0], dtype=torch.uint8),
 'dim': tensor([[  3,  15, 320, 320,   1,   1,   1,   1]], dtype=torch.int16),
 'intent_p1': tensor([0.]),
 'intent_p2': tensor([0.]),
 'intent_p3': tensor([0.]),
 'intent_code': tensor([0], dtype=torch.int16),
 'datatype': tensor([4], dtype=torch.int16),
 'bitpix': tensor([16], dtype=torch.int16),
 'slice_start': tensor([0], dtype=torch.int16),
 'pixdim': tensor([[1.0000, 4.8000, 0.8750, 0.8750, 0.0000, 0.0000, 0.0000, 0.0000]]),
 'vox_offset': tensor([0.]),
 'scl_slope': tensor([nan]),
 'scl_inter': tensor([nan]),
 'slice_end': tensor([0], dtype=torch.int16),
 'slice_code': tensor([0], dtype=torch.uint8),
 'xyzt_units': tensor([2], dtype=torch.uint8),
 'cal_max': tensor([0.]),
 'cal_min': tensor([0.]),
 'slice_duration': tensor([0.]),
 'toffset': tensor([0.]),
 'glmax': tensor([0], dtype=torch.int32

In [12]:
label.shape

(1, 5, 39, 448, 476)

In [14]:
import numpy as np

In [15]:
np.sum(label[0, 4])

117387.0